# Congressional Voting Dataset

In [56]:
import pandas as pd
import matplotlib.pyplot as plt

## EDA

In [57]:
congress_df_train = pd.read_csv("CongressionalVotingID.shuf.lrn.csv")
congress_df_train.drop(columns=['ID'], inplace=True)

In [58]:
congress_df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   class                                   218 non-null    object
 1   handicapped-infants                     218 non-null    object
 2   water-project-cost-sharing              218 non-null    object
 3   adoption-of-the-budget-resolution       218 non-null    object
 4   physician-fee-freeze                    218 non-null    object
 5   el-salvador-aid                         218 non-null    object
 6   religious-groups-in-schools             218 non-null    object
 7   anti-satellite-test-ban                 218 non-null    object
 8   aid-to-nicaraguan-contras               218 non-null    object
 9   mx-missile                              218 non-null    object
 10  immigration                             218 non-null    object
 11  synfue

In [ ]:
for i in congress_df_train.columns:
    print(congress_df_train[i].unique())

# count total no of 'unknown' values in the dataset
print(unknown_count = (congress_df_train == 'unknown').sum().sum())

['democrat' 'republican']
['y' 'n' 'unknown']
['n' 'y' 'unknown']
['y' 'n' 'unknown']
['n' 'y' 'unknown']
['y' 'n' 'unknown']
['y' 'n' 'unknown']
['y' 'n' 'unknown']
['n' 'y' 'unknown']
['y' 'n' 'unknown']
['y' 'n' 'unknown']
['n' 'y' 'unknown']
['n' 'y' 'unknown']
['y' 'n' 'unknown']
['y' 'n' 'unknown']
['n' 'y' 'unknown']
['unknown' 'y' 'n']


Unique values in predictors are: \
'y', 'n', 'unknown'

For decision tree and random forest classifiers, the dataset can work as it is but since for knn we need to encode 'unknown' as 0.5, so that 'unknown' distance from  'y' and 'n' is equal and the information the the person abstaining to answer is preserved, preprocessing is applied for all.

In [60]:
congress_df_train.head()

,class,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-crporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
0,democrat,y,n,y,n,y,y,y,n,y,y,n,n,y,y,n,unknown
1,democrat,n,n,y,n,y,y,n,n,n,y,y,y,y,y,n,y
2,democrat,y,n,y,n,n,n,y,y,y,n,n,n,n,n,y,unknown
3,republican,n,n,n,y,y,n,n,n,n,n,n,y,n,y,unknown,y
4,democrat,y,y,y,n,n,y,unknown,y,y,n,y,n,y,n,y,y


## Preprocessing

### Custom Label Encoding

In [61]:

def label_encoding(mapping, df):
    df_encoded = df.copy()
    for col in df_encoded.columns:
        unique_vals = set(df_encoded[col].unique())
        # Check if all unique values are in mapping keys
        if unique_vals.issubset(mapping.keys()):
            df_encoded[col] = df_encoded[col].map(mapping)
        else:
            print(f"Skipping column '{col}' — unexpected values found: {unique_vals - set(mapping.keys())}")
    return df_encoded

# Train Prep

In [62]:
# define mappings
mapping_X = {
    'y': 1, 
    'unknown': 0.5,
    'n': 0
    }

# do the encoding
congress_preprocessed_train = label_encoding(mapping_X, congress_df_train)

# export to csv
congress_preprocessed_train.to_csv("congressional_df_train_preprocessed.csv", index=False)

congress_preprocessed_train.head()

Skipping column 'class' — unexpected values found: {'republican', 'democrat'}


,class,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-crporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
0,democrat,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.5
1,democrat,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
2,democrat,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5
3,republican,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.5,1.0
4,democrat,1.0,1.0,1.0,0.0,0.0,1.0,0.5,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0


# Test prep

In [ ]:
congress_df_test = pd.read_csv("CongressionalVotingID.shuf.tes.csv")
congress_df_test.info()
# congress_df_test_ids = congress_df_test['ID']
# congress_df_test.drop(columns=['ID'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      217 non-null    int64 
 1   handicapped-infants                     217 non-null    object
 2   water-project-cost-sharing              217 non-null    object
 3   adoption-of-the-budget-resolution       217 non-null    object
 4   physician-fee-freeze                    217 non-null    object
 5   el-salvador-aid                         217 non-null    object
 6   religious-groups-in-schools             217 non-null    object
 7   anti-satellite-test-ban                 217 non-null    object
 8   aid-to-nicaraguan-contras               217 non-null    object
 9   mx-missile                              217 non-null    object
 10  immigration                             217 non-null    object
 11  synfue

## Preprocess Test Data

In [64]:
congress_preprocessed_test = label_encoding(mapping_X, congress_df_test)
congress_preprocessed_test.head()

congress_preprocessed_test.to_csv("congressional_df_test_preprocessed.csv", index=False)
congress_preprocessed_test.head()


Skipping column 'ID' — unexpected values found: {1, 2, 3, 4, 5, 7, 10, 11, 17, 19, 25, 26, 27, 29, 32, 35, 40, 47, 48, 53, 55, 56, 59, 62, 63, 64, 69, 73, 76, 78, 79, 80, 82, 83, 84, 87, 89, 90, 91, 92, 93, 94, 95, 97, 99, 100, 101, 104, 107, 108, 109, 111, 118, 119, 122, 123, 126, 127, 128, 130, 131, 133, 135, 136, 138, 139, 142, 144, 145, 151, 153, 155, 159, 161, 162, 163, 164, 165, 166, 167, 169, 176, 177, 179, 181, 183, 187, 188, 190, 191, 193, 194, 195, 198, 199, 200, 204, 205, 206, 207, 212, 214, 217, 218, 220, 222, 224, 225, 227, 229, 234, 235, 237, 238, 240, 242, 244, 246, 247, 248, 251, 252, 253, 256, 261, 262, 263, 267, 270, 271, 272, 273, 274, 278, 279, 281, 282, 285, 286, 287, 292, 299, 300, 301, 303, 305, 306, 307, 308, 311, 312, 315, 317, 318, 319, 322, 324, 325, 327, 330, 333, 335, 337, 338, 341, 342, 343, 344, 345, 349, 351, 354, 355, 356, 357, 360, 361, 366, 367, 369, 371, 372, 374, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 399, 400, 403, 404, 40

,ID,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-crporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
0,190,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,285,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.5,1.0
2,251,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
3,40,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0
4,91,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


In [65]:
# ---*--- Ignore Cell ---*---
# # label encoder for custom mapping
# def label_encoding(mapping, df):
#     df_encoded = df.copy()
#     for col in df_encoded.columns:
#         df_encoded[col] = df_encoded[col].map(mapping)
#     return df_encoded

# # define mappings
# mapping_X = {
#     'y': 1, 
#     'unknown': 0.5,
#     'n': 0
#     }

# mapping_y = {
#     'democrat': 1,
#     'republican': 0
#     }

# # prepare X and y
# X = congress_df_train.drop(columns=['class'])
# Y = congress_df_train[['class']]

# # encode X and y
# X_encoded = label_encoding(mapping_X, X)
# Y_encoded = label_encoding(mapping_y, Y)
# # Y_encoded = Y.map(mapping_y)

# # combine and export to csv (optional)
# congress_preprocessed = pd.concat([X_encoded, Y_encoded], axis=1)
# congress_preprocessed.to_csv("data/congressional-voting/congressional_voting_preprocessed.csv", index=False)